## Setup

In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

import mlflow
import dagshub
import mlflow.sklearn

In [27]:
dagshub.init(
    repo_owner="IzaKakhniashvili",
    repo_name="ML-assignment1-HousePrices",
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow"
)

Initialized MLflow to track repo "IzaKakhniashvili/ML-assignment1-HousePrices"

Repository IzaKakhniashvili/ML-assignment1-HousePrices initialized!

In [ ]:
model_name = "house_price_model"
model_uri = f"models:/{model_name}/18"
loaded_model = mlflow.sklearn.load_model(model_uri)

In [29]:
relevant_cols = ['LotFrontage', 'LotArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'ExterQual', 'BsmtQual', 'BsmtFinSF1', 'BsmtUnfSF', 'TotalBsmtSF', 'HeatingQC', '1stFlrSF', '2ndFlrSF', 'GrLivArea', 'BsmtFullBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Fireplaces', 'FireplaceQu', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'PoolArea', 'Neigh_Rank', 'Has_WoodDeckSF', 'Has_OpenPorchSF', 'Has_EnclosedPorch', 'Has_PoolArea', 'TotalSF', 'Total_Qual', 'MSZoning_RL', 'MSZoning_RM', 'Alley_Grvl', 'Alley_None', 'LotShape_IR1', 'LotShape_IR2', 'LotShape_Reg', 'LotConfig_CulDSac', 'Neighborhood_BrkSide', 'Neighborhood_Edwards', 'Neighborhood_IDOTRR', 'Neighborhood_NAmes', 'Neighborhood_NoRidge', 'Neighborhood_NridgHt', 'Neighborhood_OldTown', 'Neighborhood_Sawyer', 'Neighborhood_Somerst', 'Neighborhood_StoneBr', 'BldgType_1Fam', 'HouseStyle_1.5Fin', 'HouseStyle_2Story', 'RoofStyle_Gable', 'RoofStyle_Hip', 'RoofMatl_WdShngl', 'Exterior1st_CemntBd', 'Exterior1st_MetalSd', 'Exterior1st_VinylSd', 'Exterior1st_Wd Sdng', 'Exterior2nd_CmentBd', 'Exterior2nd_MetalSd', 'Exterior2nd_VinylSd', 'Exterior2nd_Wd Sdng', 'MasVnrType_BrkFace', 'MasVnrType_None', 'MasVnrType_Stone', 'ExterCond_Fa', 'Foundation_BrkTil', 'Foundation_CBlock', 'Foundation_PConc', 'Foundation_Slab', 'BsmtCond_Fa', 'BsmtCond_None', 'BsmtExposure_Av', 'BsmtExposure_Gd', 'BsmtExposure_No', 'BsmtExposure_None', 'BsmtFinType1_BLQ', 'BsmtFinType1_GLQ', 'BsmtFinType1_None', 'BsmtFinType1_Rec', 'BsmtFinType2_None', 'CentralAir_N', 'CentralAir_Y', 'Electrical_FuseA', 'Electrical_FuseF', 'Electrical_SBrkr', 'Functional_Typ', 'GarageType_Attchd', 'GarageType_BuiltIn', 'GarageType_Detchd', 'GarageType_None', 'GarageFinish_Fin', 'GarageFinish_None', 'GarageFinish_RFn', 'GarageFinish_Unf', 'GarageQual_Fa', 'GarageQual_None', 'GarageQual_TA', 'GarageCond_Fa', 'GarageCond_None', 'GarageCond_TA', 'PavedDrive_N', 'PavedDrive_Y', 'PoolQC_Ex', 'PoolQC_None', 'Fence_MnPrv', 'Fence_None', 'SaleType_New', 'SaleType_WD', 'SaleCondition_Normal', 'SaleCondition_Partial']

In [30]:
test_df = pd.read_csv('data/test.csv')
train_df = pd.read_csv('data/train.csv')
X_test = test_df.copy()

In [31]:
neigh_map = train_df.groupby('Neighborhood')['SalePrice'].median().sort_values()
bins = pd.qcut(neigh_map, 3, labels=[1, 2, 3]).to_dict()

In [32]:
X_test['Neigh_Rank'] = X_test['Neighborhood'].map(bins).astype(float).fillna(2.0)
X_test['TotalSF'] = X_test['TotalBsmtSF'].fillna(0) + X_test['1stFlrSF'] + X_test['2ndFlrSF']
X_test['Total_Qual'] = X_test['OverallQual'] + X_test['OverallCond']
X_test['GrLivArea'] = np.log1p(X_test['GrLivArea'])
X_test['LotArea'] = np.log1p(X_test['LotArea'])

In [ ]:
for col in ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 'PoolArea']:
    X_test[f'Has_{col}'] = (X_test[col] > 0).astype(int)

In [33]:
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
for col in ['ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC', 'FireplaceQu']:
    X_test[col] = X_test[col].fillna('None').map(qual_map)

In [34]:
from sklearn.preprocessing import StandardScaler

X_test_enc = pd.get_dummies(X_test)
X_test_final = X_test_enc.reindex(columns=relevant_cols, fill_value=0)
scaler = StandardScaler()
train_df_prepared = pd.get_dummies(train_df)

In [35]:
train_df['Neigh_Rank'] = train_df['Neighborhood'].map(bins).astype(float)
train_df['TotalSF'] = train_df['TotalBsmtSF'] + train_df['1stFlrSF'] + train_df['2ndFlrSF']
train_df['Total_Qual'] = train_df['OverallQual'] + train_df['OverallCond']
train_df['GrLivArea'] = np.log1p(train_df['GrLivArea'])
train_df['LotArea'] = np.log1p(train_df['LotArea'])
for col in ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 'PoolArea']:
    train_df[f'Has_{col}'] = (train_df[col] > 0).astype(int)
for col in ['ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC', 'FireplaceQu']:
    train_df[col] = train_df[col].fillna('None').map(qual_map)

train_final_enc = pd.get_dummies(train_df).reindex(columns=relevant_cols, fill_value=0)
scaler.fit(train_final_enc.fillna(train_final_enc.median()))


,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [36]:
X_test_scaled = scaler.transform(X_test_final.fillna(train_final_enc.median()))

log_preds = loaded_model.predict(X_test_scaled)
final_prices = np.expm1(log_preds)

submission = pd.DataFrame({"Id": test_df['Id'], "SalePrice": final_prices})
submission.to_csv('submission_final.csv', index=False)
